# Renewable Energy Forecasting – API Notebook

In this notebook I show how to interact with the project programmatically.  
The goal is to expose a simple "API" for:

- Running the feature creation pipeline (raw → processed)
- Training a baseline model and getting back metrics

I do this both by:
- Calling the command-line scripts (`make_features.py`, `train.py`), and
- Calling the underlying Python functions directly.


In [2]:
import sys
from pathlib import Path
import subprocess

PROJECT_ROOT = Path.cwd().resolve().parent  # I am inside api/
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

# Make sure I can import from the project root (for RenewableEnergy_utils, etc.)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT, SCRIPTS_DIR


(PosixPath('/work/class_project/MSML610/Fall2025/projects/UmdTask15_Fall2025_Renewable_Energy_Production'),
 PosixPath('/work/class_project/MSML610/Fall2025/projects/UmdTask15_Fall2025_Renewable_Energy_Production/scripts'))

In [3]:
def run_feature_pipeline_via_cli() -> None:
    """
    Simple wrapper that runs the feature creation script from Python.

    Equivalent to:
    python3 scripts/make_features.py
    """
    result = subprocess.run(
        ["python3", str(SCRIPTS_DIR / "make_features.py")],
        capture_output=True,
        text=True,
        check=False,
    )
    print("STDOUT:")
    print(result.stdout)
    print("STDERR:")
    print(result.stderr)


def run_training_via_cli() -> None:
    """
    Simple wrapper that runs the training script from Python.

    Equivalent to:
    python3 scripts/train.py
    """
    result = subprocess.run(
        ["python3", str(SCRIPTS_DIR / "train.py")],
        capture_output=True,
        text=True,
        check=False,
    )
    print("STDOUT:")
    print(result.stdout)
    print("STDERR:")
    print(result.stderr)


## 1. Using the command-line API

Here I expose a very simple API that wraps the existing command-line entry points.  
A user who just wants to generate features and train the baseline model can call these functions instead of running shell commands manually.


In [4]:
# Run the feature creation pipeline (raw → processed/train.csv)
run_feature_pipeline_via_cli()


STDOUT:
Saved processed training data to: /work/class_project/MSML610/Fall2025/projects/UmdTask15_Fall2025_Renewable_Energy_Production/data/processed/train.csv

STDERR:



In [5]:
# Run the training pipeline and print metrics
run_training_via_cli()


STDOUT:
Number of training samples:   3456
Number of validation samples: 864
Validation MAE:  0.4206
Validation RMSE: 0.5287

STDERR:



## 2. Using the Python-level API

Instead of only calling command-line scripts, I can also call the underlying
Python functions directly. This is useful if I want to integrate the project
into a larger application or another notebook.


In [6]:
from RenewableEnergy_utils import (
    load_raw_solar,
    add_basic_time_features,
    save_processed,
    PROCESSED_DIR,
)

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
import numpy as np


In [7]:
def run_full_pipeline_and_return_metrics(target_col: str = "energy_mwh"):
    """
    High-level convenience function that:
    1. Loads raw solar data
    2. Builds features
    3. Saves processed data
    4. Trains a Random Forest
    5. Returns MAE and RMSE on the validation set
    """
    # 1–3: feature pipeline
    df_raw = load_raw_solar()
    df_features = add_basic_time_features(df_raw, target_col=target_col)
    save_processed(df_features, filename="train_from_api.csv")

    # 4. Load the saved processed data
    path = PROCESSED_DIR / "train_from_api.csv"
    df = pd.read_csv(path)

    y = df[target_col]
    X = df.drop(columns=[target_col])
    X = X.select_dtypes(include=[np.number])

    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=0.2,
        shuffle=False,
    )

    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)

    mae = mean_absolute_error(y_valid, preds)
    rmse = mean_squared_error(y_valid, preds, squared=False)

    return {"mae": mae, "rmse": rmse}
